# 🐠 Unsupervised Clustering Experiment Workflow (Full Dataset)

This notebook contains the steps to run the clustering experiments on the whole dataset. The experiments cover different embedding methods (ResNet50, DINOv2), image processing (RGB, Grayscale), and embedding normalization, using various clustering algorithms. 

We will run the clustering algorithm first, save the results, and then immediately compute the **Adjusted Mutual Information (AMI)** score.

In [1]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 34.2 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.5/803.5 kB 10.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 MB 133.8 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 77.4 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 115.6 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 MB 145.8 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 120.0 MB/s  0:00:030:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 136.1 MB/s  0:00:020:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 61.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 169.7 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 13.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 128.

In [2]:
!pip install hdbscan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 26.4 MB/s  0:00:00

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


## 1. Data Preparation and Embedding Extraction (Setup)

### 1.1 Crop Images Around Bounding Boxes

We are running the image cropping on both the training and test data folders, consolidating all results into the respective 'cropped' and 'cropped_test' directories.

In [ ]:
# Training data
!python3 crop_bbs.py --images_dir ../dataset/train --labels_dir ../dataset/train_labels --output_dir ../dataset/cropped --class_ids 1 2 3 4 5 6 7 8 9
# Test data
!python3 crop_bbs.py --images_dir ../dataset/test --labels_dir ../dataset/test_labels --output_dir ../dataset/cropped_test --class_ids 1 2 3 4 5 6 7 8 9

# remember to grayscale the images

### 1.2 Extract Embeddings for Training Data

In [ ]:
# a) ResNet50 RGB
!python3 extract_embeddings.py --image_dir /Users/alteafogh/Documents/ITU/Research_project/Finding_A_Nemo/dataset/cropped/cropped --output_file ../embeddings_files/resnet_embeddings.npz

# b) ResNet50 Grayscale
!python3 extract_embeddings.py --image_dir /Users/alteafogh/Documents/ITU/Research_project/Finding_A_Nemo/dataset/cropped/cropped_gray --output_file ../embeddings_files/resnet_embeddings_gray.npz

# c) DINOv2 RGB
!python3 extract_embeddings_dinov2.py --image_dir /Users/alteafogh/Documents/ITU/Research_project/Finding_A_Nemo/dataset/cropped/cropped --output_file ../embeddings_files/dino_embeddings.npz

# d) DINOv2 Grayscale
!python3 extract_embeddings_dinov2.py --image_dir /Users/alteafogh/Documents/ITU/Research_project/Finding_A_Nemo/dataset/cropped/cropped_gray --output_file ../embeddings_files/dino_embeddings_gray.npz

### 1.3 Extract Embeddings for Test Data

In [ ]:
# a) ResNet50 RGB
!python3 extract_embeddings.py --image_dir /Users/alteafogh/Documents/ITU/Research_project/Finding_A_Nemo/dataset/cropped/cropped_test --output_file ../embeddings_files/resnet_embeddings_test.npz

# b) ResNet50 Grayscale
!python3 extract_embeddings.py --image_dir /Users/alteafogh/Documents/ITU/Research_project/Finding_A_Nemo/dataset/cropped/cropped_test_gray --output_file ../embeddings_files/resnet_embeddings_test_gray.npz

# c) DINOv2 RGB
!python3 extract_embeddings_dinov2.py --image_dir /Users/alteafogh/Documents/ITU/Research_project/Finding_A_Nemo/dataset/cropped/cropped_test --output_file ../embeddings_files/dino_embeddings_test.npz

# d) DINOv2 Grayscale
!python3 extract_embeddings_dinov2.py --image_dir /Users/alteafogh/Documents/ITU/Research_project/Finding_A_Nemo/dataset/cropped/cropped_test_gray --output_file ../embeddings_files/dino_embeddings_test_gray.npz

### 1.4 Merge Embeddings and Normalise

Merge all images (training and test) and then normalise the embeddings. Note: The merge command is symbolic as the actual command was not provided, but it's noted here for context.


In [ ]:
# !!merging all images and merging the files in merging.npz with filenames in there

# a) ResNet50 RGB Normalisation
!python3 embeddings_normaliser.py --input_file ../embeddings_files/resnet_embeddings.npz --output_file ../normalised_embeddings/resnet_normalised_embeddings.npz

# b) ResNet50 Grayscale Normalisation
!python3 embeddings_normaliser.py --input_file ../embeddings_files/resnet_embeddings_gray.npz --output_file ../normalised_embeddings/resnet_normalised_embeddings_gray.npz

# c) DINOv2 RGB Normalisation
!python3 embeddings_normaliser.py --input_file ../embeddings_files/dino_embeddings.npz --output_file ../normalised_embeddings/dino_normalised_embeddings.npz

# d) DINOv2 Grayscale Normalisation
!python3 embeddings_normaliser.py --input_file ../embeddings_files/dino_embeddings_gray.npz --output_file ../normalised_embeddings/dino_embeddings_gray.npz

***

## 2. K-means Clustering and AMI Score

### 2.1 K-means (Not Normalised)

i) resnet50, rgb, not normalised, 9 classes

In [2]:
!python3 k_means.py --embedding_file ../embeddings_filtered/resnet_embeddings_median.npz --n_clusters 9 --output_file ../k_means_clusters/k_means_resnet_rgb_median.npz
!python3 ami.py --clustered_file ../k_means_clusters/k_means_resnet_rgb_median.npz
# --> 0.1976

🔍 Running K-Means clustering...
✅ Saved clustered data to ../k_means_clusters/k_means_resnet_rgb_median.npz

📊 AMI Calculation Summary:
   Total samples:          25094
   Cluster noise (-1):     0
   Invalid ground truths:  0
   Used in AMI:            25094

✅ Adjusted Mutual Information (AMI): 0.3574


ii) resnet50, grayscale, not normalised, 9 classes

In [3]:
!python3 k_means.py --embedding_file ../embeddings_filtered/resnet_embeddings_gray_median.npz --n_clusters 9 --output_file ../k_means_clusters/k_means_resnet_gray_median.npz
!python3 ami.py --clustered_file ../k_means_clusters/k_means_resnet_gray_median.npz
# --> 0.2214

🔍 Running K-Means clustering...
✅ Saved clustered data to ../k_means_clusters/k_means_resnet_gray_median.npz

📊 AMI Calculation Summary:
   Total samples:          25095
   Cluster noise (-1):     0
   Invalid ground truths:  0
   Used in AMI:            25095

✅ Adjusted Mutual Information (AMI): 0.3884


iii) DINOv2 rgb, not normalised, 9 classes

In [1]:
!python3 k_means.py --embedding_file ../euclidean_filtered_embeddings/dino_embeddings_filtered_median.npz --n_clusters 9 --output_file ../k_means_clusters/k_means_dino_rgb_median.npz
!python3 ami.py --clustered_file ../k_means_clusters/k_means_dino_rgb_median.npz
# --> 0.2756

🔍 Running K-Means clustering...
✅ Saved clustered data to ../k_means_clusters/k_means_dino_rgb_median.npz

📊 AMI Calculation Summary:
   Total samples:          25727
   Cluster noise (-1):     0
   Invalid ground truths:  0
   Used in AMI:            25727

✅ Adjusted Mutual Information (AMI): 0.5190


iv) DINOv2 gray, not normalised, 9 classes

In [4]:
!python3 k_means.py --embedding_file ../embeddings_filtered/dino_embeddings_gray_median.npz --n_clusters 9 --output_file ../k_means_clusters/k_means_dino_gray_median.npz
!python3 ami.py --clustered_file ../k_means_clusters/k_means_dino_gray_median.npz
# --> 0.2430

🔍 Running K-Means clustering...
✅ Saved clustered data to ../k_means_clusters/k_means_dino_gray_median.npz

📊 AMI Calculation Summary:
   Total samples:          24615
   Cluster noise (-1):     0
   Invalid ground truths:  0
   Used in AMI:            24615

✅ Adjusted Mutual Information (AMI): 0.4861


### 2.2 K-means (Normalised)

i) resnet 50, rgb, normalised, 9 classes

In [ ]:
!python3 k_means.py --embedding_file ../embeddings_filtered/resnet_embeddings_norm_median.npz --n_clusters 9 --output_file ../k_means_clusters/k_means_resnet_rgb_normalised_median.npz
!python3 ami.py --clustered_file ../k_means_clusters/k_means_resnet_rgb_normalised_median.npz
# --> 0.1886

🔍 Running K-Means clustering...
✅ Saved clustered data to ../k_means_clusters/k_means_resnet_rgb_normalised_median.npz

📊 AMI Calculation Summary:
   Total samples:          25882
   Cluster noise (-1):     0
   Invalid ground truths:  0
   Used in AMI:            25882

✅ Adjusted Mutual Information (AMI): 0.3017


ii) resnet50, grayscale, normalised, 9 classes

In [2]:
!python3 k_means.py --embedding_file ../embeddings_filtered/resnet_embeddings_gray_norm_median.npz --n_clusters 9 --output_file ../k_means_clusters/k_means_resnet_gray_normalised_median.npz
!python3 ami.py --clustered_file ../k_means_clusters/k_means_resnet_gray_normalised_median.npz
# --> 0.2099

🔍 Running K-Means clustering...
✅ Saved clustered data to ../k_means_clusters/k_means_resnet_gray_normalised_median.npz

📊 AMI Calculation Summary:
   Total samples:          25358
   Cluster noise (-1):     0
   Invalid ground truths:  0
   Used in AMI:            25358

✅ Adjusted Mutual Information (AMI): 0.3848


iii) DINOv2, rgb, normalised, 9 classes

In [3]:
!python3 k_means.py --embedding_file ../embeddings_filtered/dino_embeddings_norm_median.npz --n_clusters 9 --output_file ../k_means_clusters/k_means_dino_rgb_normalised_median.npz
!python3 ami.py --clustered_file ../k_means_clusters/k_means_dino_rgb_normalised_median.npz
# --> 0.2531

🔍 Running K-Means clustering...
✅ Saved clustered data to ../k_means_clusters/k_means_dino_rgb_normalised_median.npz

📊 AMI Calculation Summary:
   Total samples:          25849
   Cluster noise (-1):     0
   Invalid ground truths:  0
   Used in AMI:            25849

✅ Adjusted Mutual Information (AMI): 0.4432


iv) dinov2, grayscale, normalised, 9 classes

In [4]:
!python3 k_means.py --embedding_file ../embeddings_filtered/dino_embeddings_gray_norm_median.npz --n_clusters 9 --output_file ../k_means_clusters/k_means_dino_gray_normalised_median.npz
!python3 ami.py --clustered_file ../k_means_clusters/k_means_dino_gray_normalised_median.npz
# --> 0.2435

🔍 Running K-Means clustering...
✅ Saved clustered data to ../k_means_clusters/k_means_dino_gray_normalised_median.npz

📊 AMI Calculation Summary:
   Total samples:          24813
   Cluster noise (-1):     0
   Invalid ground truths:  0
   Used in AMI:            24813

✅ Adjusted Mutual Information (AMI): 0.4856


***

## 3. Agglomerative Clustering and AMI Score

### 3.1 Agglomerative Clustering (Not Normalised)

i) Resnet 50 rgb, not normalised, 9 classes

In [2]:
!python3 agglomerative_clustering.py --embedding_file ../embeddings_filtered/resnet_embeddings_median.npz --n_clusters 9 --output_file ../filtered_clusters/agglomerative_clusters_resnet_rgb.npz
!python3 ami.py --clustered_file ../filtered_clusters/agglomerative_clusters_resnet_rgb.npz
# --> 0.2221

🔍 Running Agglomerative clustering...
✅ Saved clustered data to ../filtered_clusters/agglomerative_clusters_resnet_rgb.npz

📊 AMI Calculation Summary:
   Total samples:          25094
   Cluster noise (-1):     0
   Invalid ground truths:  0
   Used in AMI:            25094

✅ Adjusted Mutual Information (AMI): 0.0764


ii) resnet50, grayscale, not normalised, 9 classes

In [3]:
!python3 agglomerative_clustering.py --embedding_file ../embeddings_filtered/resnet_embeddings_gray_median.npz --n_clusters 9 --output_file ../filtered_clusters/agglomerative_clusters_resnet_gray.npz
!python3 ami.py --clustered_file ../filtered_clusters/agglomerative_clusters_resnet_gray.npz
# --> 0.2243

🔍 Running Agglomerative clustering...
✅ Saved clustered data to ../filtered_clusters/agglomerative_clusters_resnet_gray.npz

📊 AMI Calculation Summary:
   Total samples:          25095
   Cluster noise (-1):     0
   Invalid ground truths:  0
   Used in AMI:            25095

✅ Adjusted Mutual Information (AMI): 0.0901


iii) DINOv2 rgb, nor normalised, 9 classes

In [4]:
!python3 agglomerative_clustering.py --embedding_file ../embeddings_filtered/dino_embeddings_median.npz --n_clusters 9 --output_file ../filtered_clusters/agglomerative_clusters_dino_rgb.npz
!python3 ami.py --clustered_file ../filtered_clusters/agglomerative_clusters_dino_rgb.npz
# --> 0.2898

🔍 Running Agglomerative clustering...
✅ Saved clustered data to ../filtered_clusters/agglomerative_clusters_dino_rgb.npz

📊 AMI Calculation Summary:
   Total samples:          25727
   Cluster noise (-1):     0
   Invalid ground truths:  0
   Used in AMI:            25727

✅ Adjusted Mutual Information (AMI): 0.7158


iv) DINOv2 grayscale, not normalised, 9 classes

In [5]:
!python3 agglomerative_clustering.py --embedding_file ../embeddings_filtered/dino_embeddings_gray_median.npz --n_clusters 9 --output_file ../filtered_clusters/agglomerative_clusters_dino_gray.npz
!python3 ami.py --clustered_file ../filtered_clusters/agglomerative_clusters_dino_gray.npz
# --> 0.2916

🔍 Running Agglomerative clustering...
✅ Saved clustered data to ../filtered_clusters/agglomerative_clusters_dino_gray.npz

📊 AMI Calculation Summary:
   Total samples:          24615
   Cluster noise (-1):     0
   Invalid ground truths:  0
   Used in AMI:            24615

✅ Adjusted Mutual Information (AMI): 0.0563


### 3.2 Agglomerative Clustering (Normalised)

i) resnet 50, rgb, normalised, 9 classes

In [6]:
!python3 agglomerative_clustering.py --embedding_file ../embeddings_filtered/resnet_embeddings_norm_median.npz --n_clusters 9 --output_file ../filtered_clusters/ac_resnet_rgb_normalised.npz
!python3 ami.py --clustered_file ../filtered_clusters/ac_resnet_rgb_normalised.npz
# --> 0.2165

🔍 Running Agglomerative clustering...
✅ Saved clustered data to ../filtered_clusters/ac_resnet_rgb_normalised.npz

📊 AMI Calculation Summary:
   Total samples:          25882
   Cluster noise (-1):     0
   Invalid ground truths:  0
   Used in AMI:            25882

✅ Adjusted Mutual Information (AMI): 0.0514


ii) resnet50, gray, normalised, 9 classes

In [7]:
!python3 agglomerative_clustering.py --embedding_file ../embeddings_filtered/resnet_embeddings_gray_norm_median.npz --n_clusters 9 --output_file ../filtered_clusters/ac_resnet_gray_normalised.npz
!python3 ami.py --clustered_file ../filtered_clusters/ac_resnet_gray_normalised.npz
# --> 0.2480

🔍 Running Agglomerative clustering...
✅ Saved clustered data to ../filtered_clusters/ac_resnet_gray_normalised.npz

📊 AMI Calculation Summary:
   Total samples:          25358
   Cluster noise (-1):     0
   Invalid ground truths:  0
   Used in AMI:            25358

✅ Adjusted Mutual Information (AMI): 0.0106


iii) DINOv2, rgb, normalised, 9 classes

In [8]:
!python3 agglomerative_clustering.py --embedding_file ../embeddings_filtered/dino_embeddings_norm_median.npz --n_clusters 9 --output_file ../filtered_clusters/ac_dino_rgb_normalised.npz
!python3 ami.py --clustered_file ../filtered_clusters/ac_dino_rgb_normalised.npz
# --> 0.3267

🔍 Running Agglomerative clustering...
✅ Saved clustered data to ../filtered_clusters/ac_dino_rgb_normalised.npz

📊 AMI Calculation Summary:
   Total samples:          25849
   Cluster noise (-1):     0
   Invalid ground truths:  0
   Used in AMI:            25849

✅ Adjusted Mutual Information (AMI): 0.6484


iv) DINOv2, gray, normalised, 9 classes

In [1]:
!python3 agglomerative_clustering.py --embedding_file ../embeddings_filtered/dino_embeddings_gray_norm_median.npz --n_clusters 9 --output_file ../filtered_clusters/ac_dino_gray_normalised.npz
!python3 ami.py --clustered_file ../filtered_clusters/ac_dino_gray_normalised.npz
# --> 0.2556

python3: can't open file '/Users/alteafogh/Downloads/agglomerative_clustering.py': [Errno 2] No such file or directory
python3: can't open file '/Users/alteafogh/Downloads/ami.py': [Errno 2] No such file or directory


***

## 4. HDBSCAN Clustering and AMI Score

### 4.1 HDBSCAN (Not Normalised)

i) resnet 50, rgb, not normalised, min_cluster_size 10

In [3]:
!python3 hdbscan_clustering.py --embedding_file ../embeddings_filtered/resnet_embeddings_median.npz --min_cluster_size 10 --output_file ../filtered_clusters/hdbscan_resnet_rgb.npz
!python3 ami.py --clustered_file ../filtered_clusters/hdbscan_resnet_rgb.npz
# --> 0.0013

🔍 Running HDBSCAN clustering...
/opt/conda/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
✅ Saved clustered data to ../filtered_clusters/hdbscan_resnet_rgb.npz

📊 AMI Calculation Summary:
   Total samples:          25094
   Cluster noise (-1):     110
   Invalid ground truths:  0
   Used in AMI:            24984

✅ Adjusted Mutual Information (AMI): 0.0049


ii) resnet50, grayscale, not normalised, min_cluster_size 10

In [4]:
!python3 hdbscan_clustering.py --embedding_file ../embeddings_filtered/resnet_embeddings_gray_median.npz --min_cluster_size 10 --output_file ../filtered_clusters/hdbscan_resnet_gray.npz
!python3 ami.py --clustered_file ../filtered_clusters/hdbscan_resnet_gray.npz
# --> 0.0005

🔍 Running HDBSCAN clustering...
/opt/conda/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
✅ Saved clustered data to ../filtered_clusters/hdbscan_resnet_gray.npz

📊 AMI Calculation Summary:
   Total samples:          25095
   Cluster noise (-1):     806
   Invalid ground truths:  0
   Used in AMI:            24289

✅ Adjusted Mutual Information (AMI): 0.0339


iii) DINOv2 rgb, nor normalised, min_cluster_size 10

In [5]:
!python3 hdbscan_clustering.py --embedding_file ../embeddings_filtered/dino_embeddings_median.npz --min_cluster_size 10 --output_file ../filtered_clusters/hdbscan_dino_rgb.npz
!python3 ami.py --clustered_file ../filtered_clusters/hdbscan_dino_rgb.npz
# --> 0.3080

🔍 Running HDBSCAN clustering...
/opt/conda/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
✅ Saved clustered data to ../filtered_clusters/hdbscan_dino_rgb.npz

📊 AMI Calculation Summary:
   Total samples:          25727
   Cluster noise (-1):     14506
   Invalid ground truths:  0
   Used in AMI:            11221

✅ Adjusted Mutual Information (AMI): 0.3109


iv) DINOv2, grayscale, not normalised, min_cluster_size 10

In [6]:
!python3 hdbscan_clustering.py --embedding_file ../embeddings_filtered/dino_embeddings_gray_median.npz --min_cluster_size 10 --output_file ../filtered_clusters/hdbscan_dino_gray.npz
!python3 ami.py --clustered_file ../filtered_clusters/hdbscan_dino_gray.npz
# --> 0.3038

🔍 Running HDBSCAN clustering...
/opt/conda/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
✅ Saved clustered data to ../filtered_clusters/hdbscan_dino_gray.npz

📊 AMI Calculation Summary:
   Total samples:          24615
   Cluster noise (-1):     14400
   Invalid ground truths:  0
   Used in AMI:            10215

✅ Adjusted Mutual Information (AMI): 0.3245


### 4.2 HDBSCAN (Normalised)

i) resnet 50, rgb, normalised, min_cluster_size 10

In [7]:
!python3 hdbscan_clustering.py --embedding_file ../embeddings_filtered/resnet_embeddings_norm_median.npz --min_cluster_size 10 --output_file ../filtered_clusters/hdbscan_resnet_rgb_normalised.npz
!python3 ami.py --clustered_file ../filtered_clusters/hdbscan_resnet_rgb_normalised.npz
# --> 0.0013

🔍 Running HDBSCAN clustering...
/opt/conda/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
✅ Saved clustered data to ../filtered_clusters/hdbscan_resnet_rgb_normalised.npz

📊 AMI Calculation Summary:
   Total samples:          25882
   Cluster noise (-1):     1277
   Invalid ground truths:  0
   Used in AMI:            24605

✅ Adjusted Mutual Information (AMI): 0.0639


ii) resnet50, gray, normalised, min_cluster_size 10

In [8]:
!python3 hdbscan_clustering.py --embedding_file ../normalised_embeddings/resnet_normalised_embeddings_gray.npz --min_cluster_size 10 --output_file ../filtered_clusters/hdbscan_resnet_gray_normalised.npz
!python3 ami.py --clustered_file ../filtered_clusters/hdbscan_resnet_gray_normalised.npz
# --> 0.1129

🔍 Running HDBSCAN clustering...
/opt/conda/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
✅ Saved clustered data to ../filtered_clusters/hdbscan_resnet_gray_normalised.npz

📊 AMI Calculation Summary:
   Total samples:          38572
   Cluster noise (-1):     27583
   Invalid ground truths:  0
   Used in AMI:            10989

✅ Adjusted Mutual Information (AMI): 0.2193


iii) DINOv2, rgb, normalised, min_cluster_size 10

In [9]:
!python3 hdbscan_clustering.py --embedding_file ../embeddings_filtered/dino_embeddings_norm_median.npz --min_cluster_size 10 --output_file ../filtered_clusters/hdbscan_dino_rgb_normalised.npz
!python3 ami.py --clustered_file ../filtered_clusters/hdbscan_dino_rgb_normalised.npz
# --> 0.3045

🔍 Running HDBSCAN clustering...
/opt/conda/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
✅ Saved clustered data to ../filtered_clusters/hdbscan_dino_rgb_normalised.npz

📊 AMI Calculation Summary:
   Total samples:          25849
   Cluster noise (-1):     14521
   Invalid ground truths:  0
   Used in AMI:            11328

✅ Adjusted Mutual Information (AMI): 0.3129


iv) DINOv2, gray, normalised, min_cluster_size 10

In [10]:
!python3 hdbscan_clustering.py --embedding_file ../embeddings_filtered/dino_embeddings_gray_norm_median.npz --min_cluster_size 10 --output_file ../filtered_clusters/hdbscan_dino_gray_normalised.npz
!python3 ami.py --clustered_file ../filtered_clusters//hdbscan_dino_gray_normalised.npz
# --> 0.3026

🔍 Running HDBSCAN clustering...
/opt/conda/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
✅ Saved clustered data to ../filtered_clusters/hdbscan_dino_gray_normalised.npz

📊 AMI Calculation Summary:
   Total samples:          24813
   Cluster noise (-1):     14476
   Invalid ground truths:  0
   Used in AMI:            10337

✅ Adjusted Mutual Information (AMI): 0.3198


***

## 5. UMAP + K-means Clustering and AMI Score

### 5.1 UMAP + K-means (Not Normalised)

i) resnet, rgb, not normalised 9 classes

In [ ]:
!python3 umap_k_means.py --embedding_file ../embeddings_files/resnet_embeddings.npz --n_clusters 9 --output_file ../umap_k_means_clusters/umap_km_resnet_rgb.npz
!python3 ami.py --clustered_file ../umap_k_means_clusters/umap_km_resnet_rgb.npz
# --> 0.2477

The TensorFlow library was compiled to use AVX instructions, but these aren't available on your machine.
Traceback (most recent call last):
  File "/Users/alteafogh/Documents/ITU/aau_work/Ariel2/scripts/ami.py", line 71, in <module>
    main()
  File "/Users/alteafogh/Documents/ITU/aau_work/Ariel2/scripts/ami.py", line 55, in main
    data = np.load(args.clustered_file, allow_pickle=True)
  File "/Users/alteafogh/opt/anaconda3/lib/python3.9/site-packages/numpy/lib/npyio.py", line 405, in load
    fid = stack.enter_context(open(os_fspath(file), "rb"))
FileNotFoundError: [Errno 2] No such file or directory: '../umap_k_means_clusters/umap_km_resnet_rgb.npz'


ii) resnet grayscale, not normalised 9 classes

In [ ]:
!python3 umap_k_means.py --embedding_file ../embeddings_filtered/resnet_embeddings_gray_median.npz --n_clusters 9 --output_file ../umap_k_means_clusters/umap_km_resnet_gray.npz
!python3 ami.py --clustered_file ../umap_k_means_clusters/umap_km_resnet_gray.npz
# --> 0.2931

The TensorFlow library was compiled to use AVX instructions, but these aren't available on your machine.
^C
Traceback (most recent call last):
  File "/Users/alteafogh/Documents/ITU/aau_work/Ariel2/scripts/ami.py", line 71, in <module>
    main()
  File "/Users/alteafogh/Documents/ITU/aau_work/Ariel2/scripts/ami.py", line 55, in main
    data = np.load(args.clustered_file, allow_pickle=True)
  File "/Users/alteafogh/opt/anaconda3/lib/python3.9/site-packages/numpy/lib/npyio.py", line 405, in load
    fid = stack.enter_context(open(os_fspath(file), "rb"))
FileNotFoundError: [Errno 2] No such file or directory: '../umap_k_means_clusters/umap_km_resnet_gray.npz'


iii) dino, rgb, not normalised 9 classes

In [5]:
!python3 umap_k_means.py --embedding_file ../embeddings_filtered/dino_embeddings_median.npz --n_clusters 9 --output_file ../umap_k_means_clusters/umap_km_dino_rgb_median.npz
!python3 ami.py --clustered_file ../umap_k_means_clusters/umap_km_dino_rgb_median.npz
# --> 0.3986

The TensorFlow library was compiled to use AVX instructions, but these aren't available on your machine.
^C
Traceback (most recent call last):
  File "/Users/alteafogh/Documents/ITU/aau_work/Ariel2/scripts/ami.py", line 71, in <module>
    main()
  File "/Users/alteafogh/Documents/ITU/aau_work/Ariel2/scripts/ami.py", line 55, in main
    data = np.load(args.clustered_file, allow_pickle=True)
  File "/Users/alteafogh/opt/anaconda3/lib/python3.9/site-packages/numpy/lib/npyio.py", line 405, in load
    fid = stack.enter_context(open(os_fspath(file), "rb"))
FileNotFoundError: [Errno 2] No such file or directory: '../umap_k_means_clusters/umap_km_dino_rgb_median.npz'


iv) dino, gray, not normalised 9 classes

In [ ]:
!python3 umap_k_means.py --embedding_file ../embeddings_files/dino_embeddings_gray.npz --n_clusters 9 --output_file ../umap_k_means_clusters/umap_km_dino_gray.npz
!python3 ami.py --clustered_file ../umap_k_means_clusters/umap_km_dino_gray.npz
# --> 0.3654

### 5.2 UMAP + K-means (Normalised)

i) resnet, rgb, normalised 9 classes

In [ ]:
!python3 umap_k_means.py --embedding_file ../normalised_embeddings/resnet_normalised_embeddings.npz --n_clusters 9 --output_file ../umap_k_means_clusters/umap_km_resnet_normalised_rgb.npz
!python3 ami.py --clustered_file ../umap_k_means_clusters/umap_km_resnet_normalised_rgb.npz
# --> 0.2892

ii) resnet grayscale, normalised 9 classes

In [ ]:
!python3 umap_k_means.py --embedding_file ../normalised_embeddings/resnet_normalised_embeddings_gray.npz --n_clusters 9 --output_file ../umap_k_means_clusters/umap_km_resnet_normalised_gray.npz
!python3 ami.py --clustered_file ../umap_k_means_clusters/umap_km_resnet_normalised_gray.npz
# --> 0.3088

iii) dino, rgb, normalised 9 classes

In [ ]:
!python3 umap_k_means.py --embedding_file ../normalised_embeddings/dino_normalised_embeddings.npz --n_clusters 9 --output_file ../umap_k_means_clusters/umap_km_dino_normalised_rgb.npz
!python3 ami.py --clustered_file ../umap_k_means_clusters/umap_km_dino_normalised_rgb.npz
# --> 0.4107

iv) dino, gray, normalised 9 classes

In [ ]:
!python3 umap_k_means.py --embedding_file ../normalised_embeddings/dino_normalised_embeddings_gray.npz --n_clusters 9 --output_file ../umap_k_means_clusters/umap_km_dino_normalised_gray.npz
!python3 ami.py --clustered_file ../umap_k_means_clusters/umap_km_dino_normalised_gray.npz
# --> 0.3739

***

## 6. UMAP + Agglomerative Clustering and AMI Score

### 6.1 UMAP + Agglomerative Clustering (Not Normalised)

i) resnet, rgb, not normalised 9 classes

In [ ]:
!python3 umap_ac.py --embedding_file ../embeddings_files/resnet_embeddings.npz --n_clusters 9 --output_file ../umap_ac_clusters/umap_ac_resnet_rgb.npz
!python3 ami.py --clustered_file ../umap_ac_clusters/umap_ac_resnet_rgb.npz
# --> 0.2516

ii) resnet grayscale, not normalised 9 classes

In [ ]:
!python3 umap_ac.py --embedding_file ../embeddings_filtered/resnet_embeddings_gray_median.npz --n_clusters 9 --output_file ../umap_ac_clusters/umap_ac_resnet_gray.npz
!python3 ami.py --clustered_file ../umap_ac_clusters/umap_ac_resnet_gray.npz
# --> 0.3055

iii) dino, rgb, not normalised 9 classes

In [ ]:
!python3 umap_ac.py --embedding_file ../embeddings_files/dino_embeddings.npz --n_clusters 9 --output_file ../umap_ac_clusters/umap_ac_dino_rgb.npz
!python3 ami.py --clustered_file ../umap_ac_clusters/umap_ac_dino_rgb.npz
# --> 0.3765

iv) dino, gray, not normalised 9 classes

In [ ]:
!python3 umap_ac.py --embedding_file ../embeddings_files/dino_embeddings_gray.npz --n_clusters 9 --output_file ../umap_ac_clusters/umap_ac_dino_gray.npz
!python3 ami.py --clustered_file ../umap_ac_clusters/umap_ac_dino_gray.npz
# --> 0.3441

### 6.2 UMAP + Agglomerative Clustering (Normalised)

i) resnet, rgb, normalised 9 classes

In [ ]:
!python3 umap_ac.py --embedding_file ../normalised_embeddings/resnet_normalised_embeddings.npz --n_clusters 9 --output_file ../umap_ac_clusters/umap_ac_resnet_normalised_rgb.npz
!python3 ami.py --clustered_file ../umap_ac_clusters/umap_ac_resnet_normalised_rgb.npz
# --> 0.2799

ii) resnet grayscale, normalised 9 classes

In [ ]:
!python3 umap_ac.py --embedding_file ../normalised_embeddings/resnet_normalised_embeddings_gray.npz --n_clusters 9 --output_file ../umap_ac_clusters/umap_ac_resnet_normalised_gray.npz
!python3 ami.py --clustered_file ../umap_ac_clusters/umap_ac_resnet_normalised_gray.npz
# --> 0.3116

iii) dino, rgb, normalised 9 classes

In [ ]:
!python3 umap_ac.py --embedding_file ../normalised_embeddings/dino_normalised_embeddings.npz --n_clusters 9 --output_file ../umap_ac_clusters/umap_ac_dino_normalised_rgb.npz
!python3 ami.py --clustered_file ../umap_ac_clusters/umap_ac_dino_normalised_rgb.npz
# --> 0.3895

iv) dino, gray, normalised 9 classes

In [ ]:
!python3 umap_ac.py --embedding_file ../normalised_embeddings/dino_normalised_embeddings_gray.npz --n_clusters 9 --output_file ../umap_ac_clusters/umap_ac_dino_normalised_gray.npz
!python3 ami.py --clustered_file ../umap_ac_clusters/umap_ac_dino_normalised_gray.npz
# --> 0.3424

***

## 7. UMAP + HDBSCAN Clustering and AMI Score

### 7.1 UMAP + HDBSCAN (Not Normalised)

i) resnet, rgb, not normalised, min_cluster_size 10

In [ ]:
!python3 umap_hdbscan.py --embedding_file ../embeddings_files/resnet_embeddings.npz --min_cluster_size 10 --output_file ../umap_hdbscan_clusters/umap_hdbscan_resnet_rgb.npz
!python3 ami.py --clustered_file ../umap_hdbscan_clusters/umap_hdbscan_resnet_rgb.npz
# --> 0.2228

ii) resnet grayscale, not normalised, min_cluster_size 10

In [ ]:
!python3 umap_hdbscan.py --embedding_file ../embeddings_filtered/resnet_embeddings_gray_median.npz --min_cluster_size 10 --output_file ../umap_hdbscan_clusters/umap_hdbscan_resnet_gray.npz
!python3 ami.py --clustered_file ../umap_hdbscan_clusters/umap_hdbscan_resnet_gray.npz
# --> 0.2087

iii) dino, rgb, not normalised, min_cluster_size 10

In [ ]:
!python3 umap_hdbscan.py --embedding_file ../embeddings_files/dino_embeddings.npz --min_cluster_size 10 --output_file ../umap_hdbscan_clusters/umap_hdbscan_dino_rgb.npz
!python3 ami.py --clustered_file ../umap_hdbscan_clusters/umap_hdbscan_dino_rgb.npz
# --> 0.2356

iv) dino, gray, not normalised, min_cluster_size 10

In [ ]:
!python3 umap_hdbscan.py --embedding_file ../embeddings_files/dino_embeddings_gray.npz --min_cluster_size 10 --output_file ../umap_hdbscan_clusters/umap_hdbscan_dino_gray.npz
!python3 ami.py --clustered_file ../umap_hdbscan_clusters/umap_hdbscan_dino_gray.npz
# --> 0.2296

### 7.2 UMAP + HDBSCAN (Normalised)

i) resnet, rgb, normalised, min_cluster_size 10

In [ ]:
!python3 umap_hdbscan.py --embedding_file ../normalised_embeddings/resnet_normalised_embeddings.npz --min_cluster_size 10 --output_file ../umap_hdbscan_clusters/umap_hdbscan_resnet_normalised_rgb.npz
!python3 ami.py --clustered_file ../umap_hdbscan_clusters/umap_hdbscan_resnet_normalised_rgb.npz
# --> 0.2260

ii) resnet grayscale, normalised, min_cluster_size 10

In [ ]:
!python3 umap_hdbscan.py --embedding_file ../normalised_embeddings/resnet_normalised_embeddings_gray.npz --min_cluster_size 10 --output_file ../umap_hdbscan_clusters/umap_hdbscan_resnet_normalised_gray.npz
!python3 ami.py --clustered_file ../umap_hdbscan_clusters/umap_hdbscan_resnet_normalised_gray.npz
# --> 0.2136

iii) dino, rgb, normalised, min_cluster_size 10

In [ ]:
!python3 umap_hdbscan.py --embedding_file ../normalised_embeddings/dino_normalised_embeddings.npz --min_cluster_size 10 --output_file ../umap_hdbscan_clusters/umap_hdbscan_dino_normalised_rgb.npz
!python3 ami.py --clustered_file ../umap_hdbscan_clusters/umap_hdbscan_dino_normalised_rgb.npz
# --> 0.2321

iv) dino, gray, normalised, min_cluster_size 10

In [ ]:
!python3 umap_hdbscan.py --embedding_file ../normalised_embeddings/dino_normalised_embeddings_gray.npz --min_cluster_size 10 --output_file ../umap_hdbscan_clusters/umap_hdbscan_dino_normalised_gray.npz
!python3 ami.py --clustered_file ../umap_hdbscan_clusters/umap_hdbscan_dino_normalised_gray.npz
# --> 0.2282